In [80]:
from typing import List
import drawsvg as dw
import json
import pandas as pd

In [81]:
class Point:
    def __init__(self, p:List[float]=None, movement:str=None):
        if isinstance(p,list):
            if len(p) == 2:
                self.p1 = float(p[0])
                self.p2 = float(p[1])
            elif len(p) == 1:
                self.p1 = float(p[0])
        else:
            try:
                self.p1 = float(p)
            except:
                print(f"No point inputted for movement:{movement}")
        

        self.movement=movement

    def __str__(self):
        cls = self.__class__.__name__
        try:
            return f"{cls}(p1={self.p1}, p2={self.p2}, movement={self.movement})"
        except:
            return f"{cls}(p1={self.p1}, movement={self.movement})"
        
    def __repr__(self):
        cls = self.__class__.__name__
        try:
            return f"{cls}(p1={self.p1}, p2={self.p2}, movement={self.movement})"
        except:
            return f"{cls}(p1={self.p1}, movement={self.movement})"

    def create_line(self, path, size):
        if self.movement == 'M':
            return path.M(self.p1*size,self.p2*size)
        elif self.movement == 'L':
            return path.L(self.p1*size,self.p2*size)
        elif self.movement == 'V':
            return path.V(self.p1*size)
        elif self.movement == 'H':
            return path.H(self.p1*size)
        elif self.movement == 'Z':
            return path.Z()

In [82]:
class Shape:
    def __init__(self, points:Point=None, path=None,stroke='black', fill='none', stroke_width=1):
        self.points = [points] if not isinstance(points,list) else points
        self.path = path if path is not None else dw.Path(stroke=stroke, stroke_width=stroke_width)
        self.stroke = stroke
        self.stroke_width= stroke_width
        self.fill = fill
        self.group = dw.Group(fill=fill)

    def create_shape(self, size):
        for point in self.points:
            self.path = point.create_line(self.path, size=size)

        self.group.append(self.path)
        
        return self.group

    def __str__(self):
        return "\n".join(str(point) for point in self.points)
    
    def __repr__(self):
        cls = self.__class__.__name__
        return f"{cls}(points=[{self.points[0]}...], path={self.path} stroke={self.stroke}, fill={self.fill})"


In [83]:
class Map:
    def __init__(self, shapes:Shape=None, map_name='map',width=300, height=300, origin='center', size_factor=None):
        self.shapes = [shapes] if not isinstance(shapes,list) else shapes
        self.map = dw.Drawing(width=width, height=height, origin=origin)
        self.map_name = map_name
        self.size_factor = size_factor

    def create_map(self, size=1):
        if self.size_factor is not None:
            size = self.size_factor
        
        for shape in self.shapes:
            self.map.append(shape.create_shape(size=size))

    def save_map(self, file_name:str=None, file_path=None ):
        self.map
        if file_name is None:
            if '.svg' in self.map_name:
                file_name = self.map_name
            else:
                file_name = self.map_name + '.svg'

        if file_path is not None:
            save_path = r'/'.join([file_path, file_name])
            self.map.save_svg(save_path)
        else:
            self.map.save_svg(file_name)

In [84]:
def process_points(data):
    current_movement = None
    current_points = []
    points = []

    for item in data:
        if item.isalpha():  # Check if the item is a letter
            if current_movement is not None:
                points.append(Point(current_points,current_movement))

            current_movement = item
            current_points = []
        else:  # The item is a number
            current_points.append(item)

    if current_movement is not None:
        if current_movement != 'Z':
            points.append(Point(current_points,current_movement))
        else:
            points.append(Point(movement=current_movement))

    return points

def process_instructions(data: List[str]):
    result = []
    temp_list = []
    for item in data:
        temp_list.append(item)
        if item == 'Z':
            result.append(temp_list)
            temp_list = []
    return result


def process_shapes(data:List[list], fill='none', stroke_width=1):
    shapes = []
    for shape in data:
        shapes.append(Shape(process_points(shape), fill=fill, stroke_width=stroke_width))

    return shapes

def process_map(data, width=300, height=300, origin='center', size_factor=None):
    instructions = process_instructions(data)
    shapes = process_shapes(instructions)
    map_shape = Map(shapes=shapes, width=width, height=height, origin=origin, size_factor=size_factor)
    return map_shape

In [85]:
with open (r'C:\Users\Neil\Documents\Projects\Electoral-PH\Electoral-PH\image_creator\static\provinces.json') as file:
        provinces = json.load(file)

In [86]:
master_df = pd.read_csv(r'C:\Users\Neil\Documents\Projects\Electoral-PH\Electoral-PH\image_creator\static\data\master_data.csv', index_col=0)

In [87]:
master_df = master_df.astype({'year':str})
master_df['votes'] = pd.to_numeric(master_df['votes'])
master_df.head()

,year,position,province,color,candidate,votes
0,2004,President,Abra,#B0E0E6,Gloria_Arroyo,32644
1,2004,President,Agusan del Norte,#B0E0E6,Gloria_Arroyo,138402
2,2004,President,Agusan del Sur,#B0E0E6,Gloria_Arroyo,100998
3,2004,President,Aklan,#B0E0E6,Gloria_Arroyo,87197
4,2004,President,Albay,#B0E0E6,Gloria_Arroyo,172777


In [88]:
df = master_df.loc[(master_df['year']=='2004') &( master_df['position']=='President')]

In [89]:
df.loc[df['votes'] == "Quezon City"]

,year,position,province,color,candidate,votes


In [90]:
df.head()

,year,position,province,color,candidate,votes
0,2004,President,Abra,#B0E0E6,Gloria_Arroyo,32644
1,2004,President,Agusan del Norte,#B0E0E6,Gloria_Arroyo,138402
2,2004,President,Agusan del Sur,#B0E0E6,Gloria_Arroyo,100998
3,2004,President,Aklan,#B0E0E6,Gloria_Arroyo,87197
4,2004,President,Albay,#B0E0E6,Gloria_Arroyo,172777


In [106]:
df.loc[df['province'] == "Quezon City"]

,year,position,province,color,candidate,votes
162,2004,President,Quezon City,#0038A8,Fernando_Poe_Jr,258382


In [104]:
df.groupby('province')['votes'].idxmax()['Quezon City']

162

In [105]:
idx = df.groupby('province')['votes'].idxmax()
df = df.loc[idx]

In [94]:
idx['Quezon City']

162

In [95]:
df_max.loc[df_max['province'] == "Quezon City"]

,year,position,province,color,candidate,votes
162,2004,President,Quezon City,#0038A8,Fernando_Poe_Jr,258382


In [96]:
df.loc[df['province'] == "Quezon City"]

,year,position,province,color,candidate,votes
61,2004,President,Quezon City,#B0E0E6,Gloria_Arroyo,223768
162,2004,President,Quezon City,#0038A8,Fernando_Poe_Jr,258382
263,2004,President,Quezon City,#C4C4FF,Panfilo_Lacson,161053
364,2004,President,Quezon City,#9683EC,Raul_Roco,61169
465,2004,President,Quezon City,#3F9727,Eddie_Villanueva,78195


In [107]:
def create_instructions(data:list):
    instructions = data.strip()
    return instructions.split(" ")

In [108]:
data = []
for index, row in df.iterrows():
    province = row['province']
    instructions = create_instructions(provinces[province])

    data.append({'province': province, 'color':row['color'], 'instructions': instructions})

data.append({'province': 'Metro Manila', 'color': "none",'instructions': create_instructions(provinces['Metro Manila'])})
data.append({'province': 'Box', 'color': "none", 'instructions': create_instructions(provinces['Box'])})

In [109]:
shapes = []
for datum in data:
    instructions = process_instructions(datum['instructions'])
    shapes.extend(process_shapes(instructions, datum['color'], 0.5))

map_2004 = Map(shapes=shapes, width=800, height=1000, size_factor=3)

No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point inputted for movement:Z
No point i

In [110]:
map_2004.create_map()

In [111]:
map_2004.save_map(file_name="2004_presidential.svg")